# 02 — Qwen3.5-4B — Fine-tuning com QLoRA

Este notebook reproduz, de forma organizada para o repositório, o experimento de ajuste fino do **Qwen3.5-4B**.

A execução é dividida em quatro etapas:

1. preparação do conjunto supervisionado e reutilização do split fixo por autor;
2. treinamento QLoRA;
3. inferência nos 215 autores do conjunto de teste;
4. avaliação com o mesmo protocolo semântico usado nos demais experimentos.

Configuração:

- modelo base: `Qwen/Qwen3.5-4B`;
- 1.431 autores no total;
- 1.001 treino / 215 validação / 215 teste;
- seed 42;
- máximo de 50 publicações por autor;
- limite de entrada: 8.192 tokens;
- alvo de treinamento: até 1.024 tokens;
- 3 épocas;
- learning rate `2e-4`;
- weight decay `0.01`;
- warmup ratio `0.03`;
- micro-batch de treino `1`;
- gradient accumulation `4`;
- batch efetivo `4`;
- QLoRA `r=16`, `alpha=32`, `dropout=0.05`;
- módulos LoRA: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`;
- early stopping com paciência 2;
- inferência com `do_sample=True`, `temperature=0.7`, `top_p=0.8`, `top_k=20`, `min_p=0.0`;
- `enable_thinking=False`;
- 30 tags por pesquisador;
- SBERT `paraphrase-multilingual-mpnet-base-v2`;
- matching global greedy 1-para-1 com limiar 0,75.


## 1. Configuração do projeto

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src").exists()), current)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 2. Arquivos de entrada e diretórios de saída

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
SPLITS_DIR = DATA_DIR / "splits"

DOCUMENTS_FILE = DATA_DIR / "filtered_documents.json"
QRELS_FILE = GROUND_TRUTH_DIR / "LExR-prof-qrels_filtrado"
DOCUMENT_PROFILES_FILE = DATA_DIR / "perfis_documento_qwen.json"
SPLIT_FILE = SPLITS_DIR / "split_autores_seed42.json"

SHARED_DATA_DIR = PROJECT_ROOT / "results" / "finetuning" / "shared_data"
EXPERIMENT_DIR = PROJECT_ROOT / "results" / "finetuning" / "qwen3_5_4b"
ADAPTER_DIR = EXPERIMENT_DIR / "adapter_qlora_final"

TAGS_FILE = EXPERIMENT_DIR / "tags_brutas.json"
RANKING_FILE = EXPERIMENT_DIR / "ranking_tags.json"
INFERENCE_CHECKPOINT = EXPERIMENT_DIR / "tags_brutas_checkpoint.json"

METRICS_FILE = EXPERIMENT_DIR / "metricas_por_autor.csv"
MATCHING_FILE = EXPERIMENT_DIR / "avaliacoes_gerais.csv"
AGGREGATED_FILE = EXPERIMENT_DIR / "metricas_agregadas.csv"
SIM_DIR = EXPERIMENT_DIR / "sim_matrices"

SHARED_DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "filtered_documents.json": DOCUMENTS_FILE,
    "LExR-prof-qrels_filtrado": QRELS_FILE,
    "perfis_documento_qwen.json": DOCUMENT_PROFILES_FILE,
}.items():
    print(f"{name:35s} -> {'OK' if path.exists() else 'não encontrado'}")

## 3. Preparação do conjunto supervisionado

O split é realizado exclusivamente por autor. O mesmo arquivo de split deve ser compartilhado pelos dois experimentos de fine-tuning para garantir que treino, validação e teste sejam idênticos.

In [ ]:
prepare_cmd = [
    sys.executable, "-m", "src.finetuning.prepare_dataset",
    "--documents", str(DOCUMENTS_FILE),
    "--qrels", str(QRELS_FILE),
    "--output-dir", str(SHARED_DATA_DIR),
    "--split-path", str(SPLIT_FILE),
    "--seed", "42",
    "--max-publicacoes", "50",
    "--expected-authors", "1431",
    "--expected-train", "1001",
    "--expected-valid", "215",
    "--expected-test", "215",
]

subprocess.run(prepare_cmd, cwd=PROJECT_ROOT, check=True)

## 4. Conferência do split

In [ ]:
with SPLIT_FILE.open("r", encoding="utf-8") as f:
    split = json.load(f)

assert len(split["train"]) == 1001
assert len(split["validation"]) == 215
assert len(split["test"]) == 215

assert not (set(split["train"]) & set(split["validation"]))
assert not (set(split["train"]) & set(split["test"]))
assert not (set(split["validation"]) & set(split["test"]))

print({k: len(split[k]) for k in ("train", "validation", "test")})

## 5. Treinamento QLoRA

In [ ]:
train_cmd = [
    sys.executable, "-m", "src.finetuning.qlora",
    "--data-dir", str(SHARED_DATA_DIR),
    "--split-path", str(SPLIT_FILE),
    "--model", "Qwen/Qwen3.5-4B",
    "--output-dir", str(EXPERIMENT_DIR),
    "--seed", "42",
    "--max-input-tokens", "8192",
    "--max-target-tokens", "1024",
    "--compute-dtype", "bf16",
    "--lora-r", "16",
    "--lora-alpha", "32",
    "--lora-dropout", "0.05",
    "--lora-target-modules", "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
    "--train-micro-batch", "1",
    "--grad-accumulation", "4",
    "--eval-micro-batch", "1",
    "--epochs", "3",
    "--learning-rate", "2e-4",
    "--weight-decay", "0.01",
    "--warmup-ratio", "0.03",
    "--logging-steps", "10",
    "--eval-steps", "50",
    "--save-steps", "50",
    "--save-total-limit", "3",
    "--patience", "2",
    "--optim", "paged_adamw_8bit",
    "--scheduler", "linear",
    "--max-grad-norm", "1.0",
    "--expected-train", "1001",
    "--expected-valid", "215",
]

subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)

## 6. Inferência no conjunto de teste

In [ ]:
inference_cmd = [
    sys.executable, "-m", "src.finetuning.finetuned_inference",
    "--data-dir", str(SHARED_DATA_DIR),
    "--split-path", str(SPLIT_FILE),
    "--model", "Qwen/Qwen3.5-4B",
    "--adapter", str(ADAPTER_DIR),
    "--output", str(TAGS_FILE),
    "--checkpoint", str(INFERENCE_CHECKPOINT),
    "--ranking-output", str(RANKING_FILE),
    "--expected-test", "215",
    "--compute-dtype", "bf16",
    "--seed", "42",
    "--batch-size", "4",
    "--checkpoint-every-authors", "12",
    "--n-tags", "30",
    "--max-input-tokens", "8192",
    "--max-new-tokens", "1024",
    "--do-sample",
    "--temperature", "0.7",
    "--top-p", "0.8",
    "--top-k", "20",
    "--min-p", "0.0",
    "--repetition-penalty", "1.0",
]

subprocess.run(inference_cmd, cwd=PROJECT_ROOT, check=True)

## 7. Avaliação semântica

In [ ]:
from sentence_transformers import SentenceTransformer

from src.evaluation.semantic_matching import (
    carregar_qrels,
    construir_vocabulario,
    encodar_com_sbert,
    construir_matriz_similaridade,
    matching_greedy_1_to_1,
    salvar_npz,
    salvar_csv_avaliacoes_gerais,
)
from src.evaluation.metrics import (
    carregar_perfis_por_documento,
    avaliar_autor,
    salvar_csv_metricas,
)

SBERT_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
THRESHOLD = 0.75
THRESHOLD_COVERAGE = 0.75
TOP_K = 20

with RANKING_FILE.open("r", encoding="utf-8") as f:
    rankings_json = json.load(f)

rankings = {
    str(author): [(str(tag), float(score)) for tag, score in ranking]
    for author, ranking in rankings_json.items()
}

with SPLIT_FILE.open("r", encoding="utf-8") as f:
    split = json.load(f)

autores_teste = [str(a) for a in split["test"]]
if set(rankings) != set(autores_teste):
    raise ValueError("O ranking produzido não corresponde exatamente aos 215 autores do teste.")

gt_norm_global, gt_original_global = carregar_qrels(QRELS_FILE)
perfis_doc_global = carregar_perfis_por_documento(DOCUMENT_PROFILES_FILE)

gt_norm = {a: gt_norm_global[a] for a in autores_teste}
gt_original = {a: gt_original_global[a] for a in autores_teste}
perfis_doc = {a: perfis_doc_global.get(a, {}) for a in autores_teste}

modelo_sbert = SentenceTransformer(SBERT_MODEL)
vocabulario = construir_vocabulario(rankings, gt_norm, perfis_doc, top_k_pred=TOP_K)
cache_emb = encodar_com_sbert(modelo_sbert, vocabulario, batch_size=256)

dados_por_autor = {}
matching_por_autor = {}
metricas_por_autor = {}
SIM_DIR.mkdir(parents=True, exist_ok=True)

for autor in autores_teste:
    dados = construir_matriz_similaridade(
        autor,
        rankings[autor],
        gt_norm[autor],
        cache_emb,
        top_k=TOP_K,
    )
    if dados is None:
        continue

    matched_weights, matched_idx, matched_sims = matching_greedy_1_to_1(
        dados["sim"],
        dados["gold_weights"],
        theta=THRESHOLD,
    )

    salvar_npz(
        dados,
        matched_idx,
        matched_weights,
        matched_sims,
        SIM_DIR,
    )

    dados_por_autor[autor] = dados
    matching_por_autor[autor] = {
        "matched_weights": matched_weights,
        "matched_idx": matched_idx,
        "matched_sims": matched_sims,
    }

    docs_autor = perfis_doc.get(autor, {})
    metricas_por_autor[autor] = avaliar_autor(
        dados,
        matched_weights,
        docs_autor,
        docs_autor,
        cache_emb,
        theta_cov=THRESHOLD_COVERAGE,
    )

print(f"Autores avaliados: {len(metricas_por_autor)}")

## 8. Salvamento das métricas

In [ ]:
salvar_csv_metricas(
    metricas_por_autor,
    METRICS_FILE,
    modelo="Qwen3.5-4B-QLoRA",
)

salvar_csv_avaliacoes_gerais(
    dados_por_autor,
    matching_por_autor,
    gt_original,
    MATCHING_FILE,
    modelo="Qwen3.5-4B-QLoRA",
    rank_max=20,
)

metric_names = list(next(iter(metricas_por_autor.values())).keys())
medias = {
    name: float(np.mean([m[name] for m in metricas_por_autor.values()]))
    for name in metric_names
}
pd.DataFrame([medias]).to_csv(AGGREGATED_FILE, index=False)

pd.DataFrame([medias])

## 9. Saídas do experimento

Os principais artefatos produzidos são:

- adaptador QLoRA final;
- checkpoints de treinamento;
- logs e curva de loss;
- tags brutas dos 215 autores de teste;
- ranking de tags;
- métricas por autor;
- métricas agregadas;
- CSV detalhado do matching;
- matrizes de similaridade `.npz`.

A avaliação utiliza exatamente o mesmo protocolo semântico aplicado às demais abordagens.